# Paper 49 backbone on YOLO26n: LIAM, MMB and AConv (LUMA-YOLO backbone)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 49:** **LIAM** in front of layer 0, **MMB** inside the C3k2 at layers 2, 4, 6, 8, **AConv** at the stride-2 Conv of layers 1, 3, 5, 7 |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (SPPELAN, C2PSA removal and SimAM of Paper 49 are not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

Five variants are trained (one module at a time, all three, and the LIAM + MMB pair), so the final table shows each module separately:

| Run | Backbone |
|---|---|
| `bb_aconv` | AConv at layers 1, 3, 5, 7 |
| `bb_mmb` | MMB inside the C3k2 at layers 2, 4, 6, 8 |
| `bb_liam` | LIAM in front of layer 0 |
| `bb_luma` | LIAM + MMB + AConv (the Paper 49 backbone) |
| `bb_liam_mmb` | LIAM + MMB (no AConv; the best pair in the Paper 49 ablation) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so luma_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 49 backbone: LIAM, MMB, AConv

**AConv** (ultralytics built-in, Paper 49 Section 3.1): 2×2 average pool (stride 1) → the same 3×3 stride-2 conv.
It smooths noisy local responses before down-sampling.

**MMB — Memory-Modulated Bottleneck** (Paper 49, Eq. 8–12), replaces the bottleneck / C3k inside the backbone C3k2:
- dynamic branch: X_dy = BN(Conv3×3(SiLU(BN(Conv1×1(X)))));
- memory branch: n = 4 learnable prototypes P_j (C × 1 × 1); gate G = Softmax(Conv1×1(GAP(X))); M = Σ G_j · P_j;
- fusion: β = sigmoid(α), X_fuse = β · X_dy + (1 − β) · M;  output Y = X + SiLU(X_fuse).

**LIAM — Lightweight Illumination-Adaptive Module** (Paper 49, Eq. 1–6), in front of the first conv:
- illumination network: 3×3 conv + ReLU → three experts (colour-contrast with ECA, high-detail with spatial attention,
  geometric-boundary with dilated depthwise convs d = 1, 2, 3) mixed by a softmax gating network → I = σ(Conv3×3) + X;
- reflectance R = X / I; self-calibration network (depthwise-separable conv + 3 cascaded calibration units) gives a
  calibration map C; output = X + γ · (R − C).

**Where they sit in YOLO26n.**

| Layer | Base YOLO26n | `bb_aconv` | `bb_mmb` | `bb_liam` | `bb_luma` | `bb_liam_mmb` |
|---|---|---|---|---|---|---|
| input → 0 | Conv | Conv | Conv | **LIAM** → Conv | **LIAM** → Conv | **LIAM** → Conv |
| 1, 3, 5, 7 | Conv 3×3 s2 | **AConv** | Conv | Conv | **AConv** | Conv |
| 2, 4, 6, 8 | C3k2 | C3k2 | C3k2 with **MMB** | C3k2 | C3k2 with **MMB** | C3k2 with **MMB** |
| 9, 10 | SPPF, C2PSA | unchanged | unchanged | unchanged | unchanged | unchanged |

No layer index changes, so the neck connections stay the same.

**Pretrained weights (stable start).** `yolo26n.pt` is loaded first, then the layers are changed:
- AConv: its 3×3 stride-2 conv takes the pretrained conv and BN of the replaced layer (only the 2×2 average pool is new);
- MMB: the outer C3k2 convs keep their weights; inside MMB the last BN scale and the prototypes start at 0, so each MMB
  starts as an identity (Y = X);
- LIAM: γ starts at 0, so the image reaching layer 0 is unchanged at the start.

**Not used here:** the SCI no-reference illumination loss (λ·L_sci) of Paper 49 — only the standard YOLO26 detection loss
is used, so LIAM is learned through the detection loss alone.

**Assumptions** (not stated in Paper 49): AConv at all four backbone stride-2 convs (1, 3, 5, 7); 4 memory prototypes;
3 calibration units; the residual form X + γ · (R − C) with γ = 0 at the start; I is clamped to ≥ 0.05 before the division.

In [ ]:
%%writefile luma_backbone.py
"""Paper 49 backbone for YOLO26n: LIAM (input illumination adaptation), MMB (inside the backbone C3k2), AConv (down-sampling)."""
import torch
import torch.nn as nn
import torch.nn.functional as F

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import AConv, C3k2
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

ACONV_LAYERS = (1, 3, 5, 7)
MMB_LAYERS = (2, 4, 6, 8)
N_PROTOTYPES = 4
N_CAL_UNITS = 3


def softmax_list(logits):
    """Softmax across a list of same-shaped tensors, written with 4-D ops only (NCNN friendly)."""
    mx = logits[0]
    for l in logits[1:]:
        mx = torch.maximum(mx, l)
    e = [torch.exp(l - mx) for l in logits]
    s = e[0]
    for t in e[1:]:
        s = s + t
    return [t / s for t in e]


# ---------------- MMB ----------------
class MMB(nn.Module):
    """Memory-Modulated Bottleneck: Y = X + SiLU(beta * X_dy + (1 - beta) * sum_j G_j P_j). Starts as identity."""

    def __init__(self, c, n=N_PROTOTYPES):
        super().__init__()
        self.cv1 = Conv(c, c, 1, 1)  # Conv1x1 + BN + SiLU
        self.cv2 = nn.Conv2d(c, c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)
        nn.init.zeros_(self.bn2.weight)  # X_dy = 0 at the start
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, n, 1))
        self.proto = nn.Parameter(torch.zeros(c, n, 1, 1))  # prototypes P_j as a 1x1 conv weight, 0 at the start
        self.alpha = nn.Parameter(torch.zeros(1))
        self.n = n

    def forward(self, x):
        x_dy = self.bn2(self.cv2(self.cv1(x)))
        g = self.gate(x)  # (B, n, 1, 1)
        g = torch.cat(softmax_list(list(g.split(1, 1))), 1)
        m = F.conv2d(g, self.proto)  # (B, C, 1, 1) = sum_j G_j * P_j
        beta = torch.sigmoid(self.alpha)
        return x + F.silu(beta * x_dy + (1 - beta) * m)


# ---------------- LIAM ----------------
class ECA1(nn.Module):
    """ECA with a k=3 conv across channels (4-D Conv2d form)."""

    def __init__(self, k=3):
        super().__init__()
        self.conv = nn.Conv2d(1, 1, (k, 1), padding=(k // 2, 0), bias=False)

    def forward(self, x):
        y = F.adaptive_avg_pool2d(x, 1).transpose(1, 2)
        return torch.sigmoid(self.conv(y)).transpose(1, 2)


class LIAM(nn.Module):
    """Lightweight Illumination-Adaptive Module (Retinex-style): I = H_light(X), R = X / I, out = X + gamma * (R - C(R))."""

    def __init__(self, c=3, k=N_CAL_UNITS):
        super().__init__()
        # illumination estimation network
        self.stem = nn.Sequential(nn.Conv2d(c, c, 3, 1, 1), nn.ReLU())
        self.gating = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, 3, 1))
        self.cc_dw, self.cc_eca, self.cc_pw = nn.Conv2d(c, c, 3, 1, 1, groups=c), ECA1(3), nn.Conv2d(c, c, 1)
        self.hd_dw, self.hd_sa, self.hd_pw = nn.Conv2d(c, c, 5, 1, 2, groups=c), nn.Conv2d(2, 1, 5, 1, 2), nn.Conv2d(c, c, 1)
        self.gb_pw = nn.Conv2d(c, c, 1)
        self.gb_dw = nn.ModuleList(nn.Conv2d(c, c, 3, 1, d, dilation=d, groups=c) for d in (1, 2, 3))
        self.gb_att = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c, 1), nn.ReLU(), nn.Conv2d(c, 3 * c, 1))
        self.gb_out = nn.Conv2d(c, c, 1)
        self.light = nn.Conv2d(c, c, 3, 1, 1)
        # self-calibration network
        self.cal_in = nn.Sequential(nn.Conv2d(c, c, 3, 1, 1, groups=c), nn.Conv2d(c, c, 1), nn.BatchNorm2d(c), nn.Sigmoid())
        self.units = nn.ModuleList(
            nn.Sequential(
                nn.Conv2d(c, c, 3, 1, 1, groups=c), nn.Conv2d(c, c, 1), nn.BatchNorm2d(c), nn.ReLU(),
                nn.Conv2d(c, c, 3, 1, 1, groups=c), nn.Conv2d(c, c, 1), nn.BatchNorm2d(c), nn.ReLU(),
            )
            for _ in range(k)
        )
        self.cal_out = nn.Sequential(nn.Conv2d(c, c, 3, 1, 1, groups=c), nn.Conv2d(c, c, 1), nn.Sigmoid())
        self.gamma = nn.Parameter(torch.zeros(1))  # 0 at the start: layer 0 sees the original image
        self.c = c

    def illumination(self, x):
        xs = self.stem(x)
        w = softmax_list(list(self.gating(xs).split(1, 1)))  # 3 x (B, 1, 1, 1)
        fd = self.cc_dw(xs)
        f_cc = F.relu(self.cc_pw(fd * self.cc_eca(fd))) + xs
        fl = self.hd_dw(xs)
        ms = torch.sigmoid(self.hd_sa(torch.cat((fl.mean(1, keepdim=True), fl.amax(1, keepdim=True)), 1))) * fl
        f_hd = F.relu(self.hd_pw(ms)) + xs
        xr = self.gb_pw(xs)
        v = [dw(xr) for dw in self.gb_dw]
        a = softmax_list(list(self.gb_att(v[0] + v[1] + v[2]).split(self.c, 1)))
        f_gb = F.relu(self.gb_out(a[0] * v[0] + a[1] * v[1] + a[2] * v[2])) + xs
        fuse = w[0] * f_cc + w[1] * f_hd + w[2] * f_gb
        return torch.sigmoid(self.light(fuse)) + x

    def forward(self, x):
        i = torch.clamp(self.illumination(x), min=0.05)
        r = x / i
        f = self.cal_in(r)
        for u in self.units:
            f = f + u(f)
        return x + self.gamma * (r - self.cal_out(f))


class LIAMConv(nn.Module):
    """LIAM in front of the (pretrained) first Conv of the backbone."""

    def __init__(self, conv):
        super().__init__()
        self.liam = LIAM(conv.conv.in_channels)
        self.conv = conv

    def forward(self, x):
        return self.conv(self.liam(x))


# ---------------- model surgery ----------------
def _swap(model, i, new, name):
    old = model.model[i]
    new = new.to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, name  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[i] = new


def add_aconv(model, layers=ACONV_LAYERS):
    """Replace the 3x3 stride-2 Conv at the given layers with AConv (the conv keeps the pretrained weights)."""
    for i in layers:
        old = model.model[i]
        assert isinstance(old, Conv) and old.conv.kernel_size == (3, 3) and old.conv.stride == (2, 2), f"layer {i} is not a 3x3 s2 Conv"
        new = AConv(old.conv.in_channels, old.conv.out_channels)
        new.cv1.load_state_dict(old.state_dict())
        _swap(model, i, new, "AConv")
    return model


def add_mmb(model, layers=MMB_LAYERS):
    """Replace the inner bottleneck / C3k of the C3k2 at the given layers with MMB (outer convs keep their weights)."""
    for i in layers:
        m = model.model[i]
        assert isinstance(m, C3k2), f"layer {i} is {type(m).__name__}, not C3k2"
        dev = next(m.parameters()).device
        m.m = nn.ModuleList(MMB(m.c) for _ in range(len(m.m))).to(dev)
        m.type = "C3k2_MMB"
        m.np = sum(p.numel() for p in m.parameters())
    return model


def add_liam(model):
    """Put LIAM in front of the first backbone Conv (layer 0). Returns the model."""
    old = model.model[0]
    assert isinstance(old, Conv), f"layer 0 is {type(old).__name__}, not Conv"
    _swap(model, 0, LIAMConv(old), "LIAMConv")
    return model


class LUMATrainerBase(DetectionTrainer):
    """Loads the pretrained weights first, then applies the selected Paper 49 backbone modules."""

    ACONV = MMB_ON = LIAM_ON = False

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        parts = []
        if self.LIAM_ON:
            add_liam(model)
            parts.append("LIAM before layer 0")
        if self.ACONV:
            add_aconv(model)
            parts.append(f"AConv at layers {ACONV_LAYERS}")
        if self.MMB_ON:
            add_mmb(model)
            parts.append(f"MMB in layers {MMB_LAYERS}")
        if RANK in {-1, 0}:
            print("Paper 49 backbone: " + ", ".join(parts))
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


class AConvTrainer(LUMATrainerBase):
    ACONV = True


class MMBTrainer(LUMATrainerBase):
    MMB_ON = True


class LIAMTrainer(LUMATrainerBase):
    LIAM_ON = True


class LUMATrainer(LUMATrainerBase):
    ACONV = MMB_ON = LIAM_ON = True


class LIAMMMBTrainer(LUMATrainerBase):
    MMB_ON = LIAM_ON = True


In [ ]:
import importlib, luma_backbone
importlib.reload(luma_backbone)
from luma_backbone import add_aconv, add_mmb, add_liam, AConvTrainer, MMBTrainer, LIAMTrainer, LUMATrainer, LIAMMMBTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

def first(y):
    return y[0] if isinstance(y, (list, tuple)) else y

# structure check (untrained): base vs the five Paper 49 backbones
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
variants = {
    "AConv": add_aconv(deepcopy(base)),
    "MMB": add_mmb(deepcopy(base)),
    "LIAM": add_liam(deepcopy(base)),
    "LIAM + MMB + AConv": add_mmb(add_aconv(add_liam(deepcopy(base)))),
    "LIAM + MMB": add_mmb(add_liam(deepcopy(base))),
}
with torch.no_grad():
    x = torch.rand(1, 3, 640, 640)
    yb = first(base(x))
    for tag, m in variants.items():
        m.eval()
        d = (yb - first(m(x))).abs().max()
        print(f"{tag:20s} forward OK | max |base - variant| at start = {d:.2e}"
              + ("  (0 expected: identity start)" if tag in ("MMB", "LIAM", "LIAM + MMB") else "  (small, AConv adds a 2x2 average pool)"))
    # forward check with LIAM and MMB switched on (only for this check)
    full = variants["LIAM + MMB + AConv"]
    full.model[0].liam.gamma.data.fill_(1.0)
    for mod in full.modules():
        if isinstance(mod, luma_backbone.MMB):
            mod.bn2.weight.data.fill_(1.0)
    _ = full(x)
    full.model[0].liam.gamma.data.zero_()
    for mod in full.modules():
        if isinstance(mod, luma_backbone.MMB):
            mod.bn2.weight.data.zero_()
print("layers 0-8 (full):", [variants["LIAM + MMB + AConv"].model[i].type for i in range(9)])
for tag, m in [("base", base)] + list(variants.items()):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:20s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

## 4. Training

Five Paper 49 backbone variants are trained one after another with identical settings (about 6–7.5 h on a T4 in total; the 12 h Kaggle limit still fits). If the first four were already trained in an earlier session, keep only the `bb_liam_mmb` line. To skip a variant, put `#` in front of its line in `VARIANTS`.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name, trainer)
VARIANTS = [
    ("AConv Backbone (Paper 49)", "bb_aconv", AConvTrainer),
    ("MMB Backbone (Paper 49)", "bb_mmb", MMBTrainer),
    ("LIAM Backbone (Paper 49)", "bb_liam", LIAMTrainer),
    ("LIAM + MMB + AConv Backbone (Paper 49)", "bb_luma", LUMATrainer),
    ("LIAM + MMB Backbone (Paper 49)", "bb_liam_mmb", LIAMMMBTrainer),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run, trainer in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=trainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import luma_backbone  # must be imported before loading a Paper 49 checkpoint

rows = []
for label, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper49_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_aconv_ncnn_model/`, `bb_mmb_ncnn_model/`, `bb_liam_ncnn_model/`, `bb_luma_ncnn_model/`, `bb_liam_mmb_ncnn_model/` (NCNN, imgsz 640) | Raspberry Pi 5 |
| `bb_aconv.pt`, `bb_mmb.pt`, `bb_liam.pt`, `bb_luma.pt`, `bb_liam_mmb.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `luma_backbone.py` | Jetson (needed to load the Paper 49 `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper49_backbone_edge_bench.zip paper49_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper49_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("luma_backbone.py", EDGE / "luma_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run, _ in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    try:
        p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
        shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
        print(f"✅ {run:16s} -> {run}.pt, {run}_ncnn_model/")
    except Exception as e:
        print(f"❌ {run:16s} NCNN export failed: {e}  (the .pt for Jetson is still packaged)")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import luma_backbone  # needed to load the Paper 49 checkpoints
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper49_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper49_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the five Paper 49 backbone models
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_aconv": None, "bb_mmb": None, "bb_liam": None, "bb_luma": None, "bb_liam_mmb": None}
FPS_JETSON = {"bb_aconv": None, "bb_mmb": None, "bb_liam": None, "bb_luma": None, "bb_liam_mmb": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper49_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper49_backbone_edge_bench/` | NCNN + `.pt` models, 50 bench images, `luma_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `bb_aconv_best.pt`, `bb_mmb_best.pt`, `bb_liam_best.pt`, `bb_luma_best.pt`, `bb_liam_mmb_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper49_backbone_all_results.zip"
folders = [WORK / "paper49_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run, _ in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")